
# Roxy notebook example: Charge, protonation, and approximate electrostatic density descriptors

This notebook is a **reference implementation example** for the **charge / protonation / electrostatic density descriptor family** in Roxy.

These descriptors try to summarize how a sequence behaves from the point of view of:

- ionizable residue content
- approximate protonation state
- net charge at a given pH
- charge balance and segregation
- local charge density
- terminal charge tendencies

They do **not** require any structure model. Everything is estimated directly from sequence using simple physicochemical rules and Henderson–Hasselbalch-style approximations.

## Covered outputs

This notebook implements examples such as:

- approximate net charge at different pH values
- positive and negative charge fractions
- FCR and NCPR
- protonated / deprotonated residue fractions
- charge density per residue
- terminal charge estimates
- local charge density in sliding windows
- electrostatic imbalance proxies
- acidic/basic burden
- charge asymmetry between N- and C-terminal regions
- class-style implementation for later migration into Roxy


In [1]:

import numpy as np
import pandas as pd


## Demo dataset

In [2]:

df_demo = pd.DataFrame(
    {
        "sequence_id": [
            "chg_1",
            "chg_2",
            "chg_3",
            "chg_4",
            "chg_5",
            "chg_6",
        ],
        "sequence": [
            "MKWVTFISLLFLFSSAYSRGVFRR",
            "GGGGGGGGGGGGGGG",
            "KRRKRRKRRKRRDDDDEE",
            "ACDEFGHIKLMNPQRSTVWY",
            "PPPPGSSSSSTTTTNNQQQ",
            "MSTNPKPQRITLKDGNKVELV",
        ],
        "label": ["A", "B", "A", "B", "A", "B"],
    }
)

df_demo


,sequence_id,sequence,label
0,chg_1,MKWVTFISLLFLFSSAYSRGVFRR,A
1,chg_2,GGGGGGGGGGGGGGG,B
2,chg_3,KRRKRRKRRKRRDDDDEE,A
3,chg_4,ACDEFGHIKLMNPQRSTVWY,B
4,chg_5,PPPPGSSSSSTTTTNNQQQ,A
5,chg_6,MSTNPKPQRITLKDGNKVELV,B


## Constants

In [3]:

STANDARD_AA = set("ACDEFGHIKLMNPQRSTVWY")

P_KA_SIDECHAIN = {
    "C": 8.3,
    "D": 3.9,
    "E": 4.3,
    "H": 6.0,
    "K": 10.5,
    "R": 12.5,
    "Y": 10.1,
}

P_KA_N_TERM = 9.69
P_KA_C_TERM = 2.34

POSITIVE_GROUP = set("KRH")
NEGATIVE_GROUP = set("DECY")
ACIDIC_GROUP = set("DE")
BASIC_GROUP = set("KRH")
IONIZABLE_GROUP = set("CDEHKRY")

HBOND_DONORS = {
    "A": 0, "C": 0, "D": 0, "E": 0, "F": 0,
    "G": 0, "H": 1, "I": 0, "K": 1, "L": 0,
    "M": 0, "N": 1, "P": 0, "Q": 1, "R": 1,
    "S": 1, "T": 1, "V": 0, "W": 1, "Y": 1,
}


## Helper functions

In [4]:

def clean_sequence(seq: str) -> str:
    if pd.isna(seq):
        return ""
    seq = str(seq).strip().upper().replace("*", "")
    return "".join([aa for aa in seq if aa in STANDARD_AA])


def windows(seq: str, size: int):
    if len(seq) < size:
        return []
    return [seq[i:i+size] for i in range(len(seq) - size + 1)]


def positive_fraction(seq: str) -> float:
    if len(seq) == 0:
        return np.nan
    return sum(aa in POSITIVE_GROUP for aa in seq) / len(seq)


def negative_fraction(seq: str) -> float:
    if len(seq) == 0:
        return np.nan
    return sum(aa in NEGATIVE_GROUP for aa in seq) / len(seq)


def ionizable_fraction(seq: str) -> float:
    if len(seq) == 0:
        return np.nan
    return sum(aa in IONIZABLE_GROUP for aa in seq) / len(seq)


def safe_ratio(a: float, b: float) -> float:
    if b == 0:
        return np.nan
    return a / b


def positive_charge_contribution(aa: str, ph: float) -> float:
    if aa == "K":
        return 1.0 / (1.0 + 10 ** (ph - P_KA_SIDECHAIN["K"]))
    if aa == "R":
        return 1.0 / (1.0 + 10 ** (ph - P_KA_SIDECHAIN["R"]))
    if aa == "H":
        return 1.0 / (1.0 + 10 ** (ph - P_KA_SIDECHAIN["H"]))
    return 0.0


def negative_charge_contribution(aa: str, ph: float) -> float:
    if aa == "D":
        return 1.0 / (1.0 + 10 ** (P_KA_SIDECHAIN["D"] - ph))
    if aa == "E":
        return 1.0 / (1.0 + 10 ** (P_KA_SIDECHAIN["E"] - ph))
    if aa == "C":
        return 1.0 / (1.0 + 10 ** (P_KA_SIDECHAIN["C"] - ph))
    if aa == "Y":
        return 1.0 / (1.0 + 10 ** (P_KA_SIDECHAIN["Y"] - ph))
    return 0.0


def terminal_positive_charge(ph: float) -> float:
    return 1.0 / (1.0 + 10 ** (ph - P_KA_N_TERM))


def terminal_negative_charge(ph: float) -> float:
    return 1.0 / (1.0 + 10 ** (P_KA_C_TERM - ph))


def net_charge(seq: str, ph: float = 7.0) -> float:
    seq = clean_sequence(seq)
    if len(seq) == 0:
        return np.nan

    positive = terminal_positive_charge(ph)
    negative = terminal_negative_charge(ph)

    for aa in seq:
        positive += positive_charge_contribution(aa, ph)
        negative += negative_charge_contribution(aa, ph)

    return positive - negative


def protonated_basic_fraction(seq: str, ph: float = 7.0) -> float:
    seq = clean_sequence(seq)
    if len(seq) == 0:
        return np.nan
    vals = [positive_charge_contribution(aa, ph) for aa in seq if aa in BASIC_GROUP]
    return float(np.mean(vals)) if len(vals) > 0 else np.nan


def deprotonated_acidic_fraction(seq: str, ph: float = 7.0) -> float:
    seq = clean_sequence(seq)
    if len(seq) == 0:
        return np.nan
    vals = [negative_charge_contribution(aa, ph) for aa in seq if aa in ACIDIC_GROUP]
    return float(np.mean(vals)) if len(vals) > 0 else np.nan


def charge_density(seq: str, ph: float = 7.0) -> float:
    seq = clean_sequence(seq)
    if len(seq) == 0:
        return np.nan
    return net_charge(seq, ph=ph) / len(seq)


def local_charge_profile(seq: str, window: int = 5, ph: float = 7.0):
    ws = windows(seq, window)
    if len(ws) == 0:
        return []
    return [charge_density(w, ph=ph) for w in ws]


def profile_stats(values):
    if len(values) == 0:
        return {
            "mean": np.nan,
            "std": np.nan,
            "min": np.nan,
            "max": np.nan,
            "amplitude": np.nan,
        }
    return {
        "mean": float(np.mean(values)),
        "std": float(np.std(values, ddof=0)),
        "min": float(np.min(values)),
        "max": float(np.max(values)),
        "amplitude": float(np.max(values) - np.min(values)),
    }


def terminal_segment(seq: str, side: str = "N", window: int = 10) -> str:
    seq = clean_sequence(seq)
    if side == "N":
        return seq[:window]
    if side == "C":
        return seq[-window:]
    raise ValueError("side must be 'N' or 'C'")


## Core descriptor function

In [5]:

def charge_descriptors(seq: str, ph_values=(5.0, 7.0, 9.0), local_window=5, terminal_window=10) -> dict:
    seq = clean_sequence(seq)

    out = {
        "chg_length": len(seq),
        "chg_valid_residue_count": len(seq),
    }

    if len(seq) == 0:
        return out

    pos_frac = positive_fraction(seq)
    neg_frac = negative_fraction(seq)

    out["chg_positive_fraction"] = pos_frac
    out["chg_negative_fraction"] = neg_frac
    out["chg_ionizable_fraction"] = ionizable_fraction(seq)
    out["chg_fcr"] = pos_frac + neg_frac
    out["chg_ncpr"] = pos_frac - neg_frac
    out["chg_basic_acidic_ratio"] = safe_ratio(sum(aa in BASIC_GROUP for aa in seq), sum(aa in ACIDIC_GROUP for aa in seq))
    out["chg_acidic_basic_ratio"] = safe_ratio(sum(aa in ACIDIC_GROUP for aa in seq), sum(aa in BASIC_GROUP for aa in seq))

    for ph in ph_values:
        ph_tag = str(ph).replace(".", "p")

        out[f"chg_net_charge_ph{ph_tag}"] = net_charge(seq, ph=ph)
        out[f"chg_density_ph{ph_tag}"] = charge_density(seq, ph=ph)
        out[f"chg_protonated_basic_fraction_ph{ph_tag}"] = protonated_basic_fraction(seq, ph=ph)
        out[f"chg_deprotonated_acidic_fraction_ph{ph_tag}"] = deprotonated_acidic_fraction(seq, ph=ph)

        local_profile = local_charge_profile(seq, window=local_window, ph=ph)
        stats = profile_stats(local_profile)
        out[f"chg_local_mean_ph{ph_tag}"] = stats["mean"]
        out[f"chg_local_std_ph{ph_tag}"] = stats["std"]
        out[f"chg_local_min_ph{ph_tag}"] = stats["min"]
        out[f"chg_local_max_ph{ph_tag}"] = stats["max"]
        out[f"chg_local_amplitude_ph{ph_tag}"] = stats["amplitude"]

        nterm = terminal_segment(seq, side="N", window=terminal_window)
        cterm = terminal_segment(seq, side="C", window=terminal_window)

        out[f"chg_nterm_net_ph{ph_tag}"] = net_charge(nterm, ph=ph)
        out[f"chg_cterm_net_ph{ph_tag}"] = net_charge(cterm, ph=ph)
        out[f"chg_nterm_density_ph{ph_tag}"] = charge_density(nterm, ph=ph)
        out[f"chg_cterm_density_ph{ph_tag}"] = charge_density(cterm, ph=ph)
        out[f"chg_terminal_asymmetry_ph{ph_tag}"] = out[f"chg_nterm_density_ph{ph_tag}"] - out[f"chg_cterm_density_ph{ph_tag}"]

    return out


## Functional usage on one sequence

In [6]:

example = charge_descriptors(df_demo.loc[0, "sequence"], ph_values=(5.0, 7.0, 9.0), local_window=5, terminal_window=10)
list(example.items())[:20]


[('chg_length', 24),
 ('chg_valid_residue_count', 24),
 ('chg_positive_fraction', 0.16666666666666666),
 ('chg_negative_fraction', 0.041666666666666664),
 ('chg_ionizable_fraction', 0.20833333333333334),
 ('chg_fcr', 0.20833333333333331),
 ('chg_ncpr', 0.125),
 ('chg_basic_acidic_ratio', nan),
 ('chg_acidic_basic_ratio', 0.0),
 ('chg_net_charge_ph5p0', 4.002151368453628),
 ('chg_density_ph5p0', 0.16675630701890118),
 ('chg_protonated_basic_fraction_ph5p0', 0.9999991857160033),
 ('chg_deprotonated_acidic_fraction_ph5p0', nan),
 ('chg_local_mean_ph5p0', 0.10043205082563564),
 ('chg_local_std_ph5p0', 0.13416396853485554),
 ('chg_local_min_ph5p0', 0.00043092511792308663),
 ('chg_local_max_ph5p0', 0.40043250111266326),
 ('chg_local_amplitude_ph5p0', 0.4000015759947402),
 ('chg_nterm_net_ph5p0', 1.0021594065412072),
 ('chg_cterm_net_ph5p0', 3.0021545307212887)]

## Apply charge descriptors to the full dataset

In [7]:

df_chg = pd.concat(
    [
        df_demo,
        df_demo["sequence"].apply(
            lambda x: charge_descriptors(
                x,
                ph_values=(5.0, 7.0, 9.0),
                local_window=5,
                terminal_window=10,
            )
        ).apply(pd.Series),
    ],
    axis=1,
)

df_chg.head()


,sequence_id,sequence,label,chg_length,chg_valid_residue_count,chg_positive_fraction,chg_negative_fraction,chg_ionizable_fraction,chg_fcr,chg_ncpr,...,chg_local_mean_ph9p0,chg_local_std_ph9p0,chg_local_min_ph9p0,chg_local_max_ph9p0,chg_local_amplitude_ph9p0,chg_nterm_net_ph9p0,chg_cterm_net_ph9p0,chg_nterm_density_ph9p0,chg_cterm_density_ph9p0,chg_terminal_asymmetry_ph9p0
0,chg_1,MKWVTFISLLFLFSSAYSRGVFRR,A,24.0,24.0,0.166667,0.041667,0.208333,0.208333,0.125000,...,0.061771,0.133265,-0.048628,0.365963,0.414591,0.799792,2.755909,0.079979,0.275591,-0.195612
1,chg_2,GGGGGGGGGGGGGGG,B,15.0,15.0,0.000000,0.000000,0.000000,0.000000,0.000000,...,-0.033911,0.000000,-0.033911,-0.033911,0.000000,-0.169555,-0.169555,-0.016955,-0.016955,0.000000
2,chg_3,KRRKRRKRRKRRDDDDEE,A,18.0,18.0,0.666667,0.333333,1.000000,1.000000,0.333333,...,0.387937,0.762364,-1.033901,0.959705,1.993606,9.705935,-2.201085,0.970593,-0.220109,1.190702
3,chg_4,ACDEFGHIKLMNPQRSTVWY,B,20.0,20.0,0.150000,0.200000,0.350000,0.350000,-0.050000,...,-0.020044,0.276957,-0.600638,0.166026,0.766664,-2.032844,0.756541,-0.203284,0.075654,-0.278939
4,chg_5,PPPPGSSSSSTTTTNNQQQ,A,19.0,19.0,0.000000,0.000000,0.000000,0.000000,0.000000,...,-0.033911,0.000000,-0.033911,-0.033911,0.000000,-0.169555,-0.169555,-0.016955,-0.016955,0.000000


## Inspect charge descriptor columns

In [8]:

chg_cols = [c for c in df_chg.columns if c.startswith("chg_") and c not in {"chg_length", "chg_valid_residue_count"}]
len(chg_cols), chg_cols[:18]


(49,
 ['chg_positive_fraction',
  'chg_negative_fraction',
  'chg_ionizable_fraction',
  'chg_fcr',
  'chg_ncpr',
  'chg_basic_acidic_ratio',
  'chg_acidic_basic_ratio',
  'chg_net_charge_ph5p0',
  'chg_density_ph5p0',
  'chg_protonated_basic_fraction_ph5p0',
  'chg_deprotonated_acidic_fraction_ph5p0',
  'chg_local_mean_ph5p0',
  'chg_local_std_ph5p0',
  'chg_local_min_ph5p0',
  'chg_local_max_ph5p0',
  'chg_local_amplitude_ph5p0',
  'chg_nterm_net_ph5p0',
  'chg_cterm_net_ph5p0'])

In [9]:

df_chg[
    [
        "sequence_id",
        "chg_fcr",
        "chg_ncpr",
        "chg_net_charge_ph7p0",
        "chg_density_ph7p0",
        "chg_local_amplitude_ph7p0",
        "chg_nterm_density_ph7p0",
        "chg_cterm_density_ph7p0",
        "chg_terminal_asymmetry_ph7p0",
    ]
]


,sequence_id,chg_fcr,chg_ncpr,chg_net_charge_ph7p0,chg_density_ph7p0,chg_local_amplitude_ph7p0,chg_nterm_density_ph7p0,chg_cterm_density_ph7p0,chg_terminal_asymmetry_ph7p0
0,chg_1,0.208333,0.125000,3.996865,0.166536,0.400157,0.099767,0.299718,-0.199951
1,chg_2,0.000000,0.000000,-0.002016,-0.000134,0.000000,-0.000202,-0.000202,0.000000
2,chg_3,1.000000,0.333333,6.003852,0.333547,1.998901,0.999670,-0.199518,1.199188
3,chg_4,0.350000,-0.050000,0.042839,0.002142,0.627107,-0.095636,0.099719,-0.195355
4,chg_5,0.000000,0.000000,-0.002016,-0.000106,0.000000,-0.000202,-0.000202,0.000000
5,chg_6,0.285714,0.095238,1.999818,0.095229,0.399936,0.199767,0.000014,0.199753


## Dataset-level summary

In [10]:

chg_summary = (
    df_chg[chg_cols]
    .mean(axis=0, numeric_only=True)
    .sort_values(ascending=False)
    .rename("mean_value")
    .reset_index()
    .rename(columns={"index": "descriptor"})
)

chg_summary.head(15)


,descriptor,mean_value
0,chg_net_charge_ph5p0,2.338066
1,chg_nterm_net_ph5p0,2.193578
2,chg_net_charge_ph7p0,2.006557
3,chg_nterm_net_ph7p0,2.005273
4,chg_basic_acidic_ratio,1.833333
5,chg_nterm_net_ph9p0,1.655541
6,chg_net_charge_ph9p0,1.620495
7,chg_deprotonated_acidic_fraction_ph9p0,0.999987
8,chg_deprotonated_acidic_fraction_ph7p0,0.998674
9,chg_protonated_basic_fraction_ph5p0,0.992423


## Sanity checks

In [11]:

assert "chg_fcr" in df_chg.columns
assert "chg_ncpr" in df_chg.columns
assert "chg_net_charge_ph7p0" in df_chg.columns
assert "chg_density_ph7p0" in df_chg.columns
assert "chg_local_amplitude_ph7p0" in df_chg.columns
assert "chg_terminal_asymmetry_ph7p0" in df_chg.columns
assert df_chg["chg_length"].min() > 0

print(f"Number of charge / protonation descriptor columns: {len(chg_cols)}")
print("Charge descriptor checks passed.")


Number of charge / protonation descriptor columns: 49
Charge descriptor checks passed.


## Class-style implementation closer to the real package

In [12]:

class ChargeDescriptors:
    """Example class-style charge/protonation implementation for later migration into Roxy."""

    def __init__(self, ph_values=(5.0, 7.0, 9.0), local_window=5, terminal_window=10):
        self.ph_values = tuple(ph_values)
        self.local_window = int(local_window)
        self.terminal_window = int(terminal_window)

    def transform_sequence(self, seq: str) -> dict:
        return charge_descriptors(
            seq,
            ph_values=self.ph_values,
            local_window=self.local_window,
            terminal_window=self.terminal_window,
        )

    def transform(self, sequences) -> pd.DataFrame:
        return pd.DataFrame([self.transform_sequence(seq) for seq in sequences])


chg_transformer = ChargeDescriptors(ph_values=(5.0, 7.0, 9.0), local_window=5, terminal_window=10)
chg_matrix = chg_transformer.transform(df_demo["sequence"].tolist())
chg_matrix.head()


,chg_length,chg_valid_residue_count,chg_positive_fraction,chg_negative_fraction,chg_ionizable_fraction,chg_fcr,chg_ncpr,chg_basic_acidic_ratio,chg_acidic_basic_ratio,chg_net_charge_ph5p0,...,chg_local_mean_ph9p0,chg_local_std_ph9p0,chg_local_min_ph9p0,chg_local_max_ph9p0,chg_local_amplitude_ph9p0,chg_nterm_net_ph9p0,chg_cterm_net_ph9p0,chg_nterm_density_ph9p0,chg_cterm_density_ph9p0,chg_terminal_asymmetry_ph9p0
0,24,24,0.166667,0.041667,0.208333,0.208333,0.125000,NaN,0.000000,4.002151,...,0.061771,0.133265,-0.048628,0.365963,0.414591,0.799792,2.755909,0.079979,0.275591,-0.195612
1,15,15,0.000000,0.000000,0.000000,0.000000,0.000000,NaN,NaN,0.002163,...,-0.033911,0.000000,-0.033911,-0.033911,0.000000,-0.169555,-0.169555,-0.016955,-0.016955,0.000000
2,18,18,0.666667,0.333333,1.000000,1.000000,0.333333,2.0,0.500000,6.629175,...,0.387937,0.762364,-1.033901,0.959705,1.993606,9.705935,-2.201085,0.970593,-0.220109,1.190702
3,20,20,0.150000,0.200000,0.350000,0.350000,-0.050000,1.5,0.666667,1.150666,...,-0.020044,0.276957,-0.600638,0.166026,0.766664,-2.032844,0.756541,-0.203284,0.075654,-0.278939
4,19,19,0.000000,0.000000,0.000000,0.000000,0.000000,NaN,NaN,0.002163,...,-0.033911,0.000000,-0.033911,-0.033911,0.000000,-0.169555,-0.169555,-0.016955,-0.016955,0.000000


## Merge transformer output back to the dataset

In [13]:

df_chg_class = pd.concat([df_demo, chg_matrix], axis=1)
df_chg_class.head()


,sequence_id,sequence,label,chg_length,chg_valid_residue_count,chg_positive_fraction,chg_negative_fraction,chg_ionizable_fraction,chg_fcr,chg_ncpr,...,chg_local_mean_ph9p0,chg_local_std_ph9p0,chg_local_min_ph9p0,chg_local_max_ph9p0,chg_local_amplitude_ph9p0,chg_nterm_net_ph9p0,chg_cterm_net_ph9p0,chg_nterm_density_ph9p0,chg_cterm_density_ph9p0,chg_terminal_asymmetry_ph9p0
0,chg_1,MKWVTFISLLFLFSSAYSRGVFRR,A,24,24,0.166667,0.041667,0.208333,0.208333,0.125000,...,0.061771,0.133265,-0.048628,0.365963,0.414591,0.799792,2.755909,0.079979,0.275591,-0.195612
1,chg_2,GGGGGGGGGGGGGGG,B,15,15,0.000000,0.000000,0.000000,0.000000,0.000000,...,-0.033911,0.000000,-0.033911,-0.033911,0.000000,-0.169555,-0.169555,-0.016955,-0.016955,0.000000
2,chg_3,KRRKRRKRRKRRDDDDEE,A,18,18,0.666667,0.333333,1.000000,1.000000,0.333333,...,0.387937,0.762364,-1.033901,0.959705,1.993606,9.705935,-2.201085,0.970593,-0.220109,1.190702
3,chg_4,ACDEFGHIKLMNPQRSTVWY,B,20,20,0.150000,0.200000,0.350000,0.350000,-0.050000,...,-0.020044,0.276957,-0.600638,0.166026,0.766664,-2.032844,0.756541,-0.203284,0.075654,-0.278939
4,chg_5,PPPPGSSSSSTTTTNNQQQ,A,19,19,0.000000,0.000000,0.000000,0.000000,0.000000,...,-0.033911,0.000000,-0.033911,-0.033911,0.000000,-0.169555,-0.169555,-0.016955,-0.016955,0.000000



## Suggested next refactor into the package

A clean migration path into Roxy would be:

- move pKa constants into `roxy/core/constants.py`
- move helper logic into `roxy/sequence/charge.py`
- expose a class such as `ChargeDescriptors`
- allow configurable:
  - pH values
  - local window size
  - terminal window size
  - selected charge summaries
- add tests for:
  - empty sequences
  - strongly basic sequences
  - strongly acidic sequences
  - neutral sequences
  - lower-case input
  - invalid characters removed during cleaning


## Optional export

In [14]:
# df_chg.to_csv("demo_charge_descriptors.csv", index=False)
